# Model Eval; Grade Visualisation

## How to run the grading

Grading is performed by Claude Code using `data/results/grading_spec.md` as the scoring guide.

**Prerequisite:** `data/results/model_eval.json` must exist; produced by running `04_model_eval.ipynb`.

**Run the grading** from the project root:

```
claude
```

Then paste this prompt:

```
Read `data/results/grading_spec.md` and `data/results/model_eval.json`, then grade every
record according to the criteria below. Write the results to `data/results/model_eval_grades.json`.
```

Claude reads both files, scores each record on the task-specific criteria (1–5), and writes
`data/results/model_eval_grades.json`; metadata + scores only, no raw results.

Once that file is present, run the cells below.

In [1]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

with open('../data/results/model_eval_grades_haiku_4.5.json') as f:
    records = json.load(f)

rows = []
for r in records:
    base = {
        'model': r['model'],
        'task': r['task'],
        'doc': r['doc'],
        'elapsed_s': r['elapsed_s'],
    }
    for k, v in r['scores'].items():
        if k != 'notes':
            rows.append({**base, 'criterion': k, 'score': v})

df = pd.DataFrame(rows)
print(df.shape, df['model'].nunique(), 'models')

(220, 6) 10 models


In [2]:
# Heatmap: average score per model × criterion
pivot = df.groupby(['model', 'criterion'])['score'].mean().unstack()

# Order criteria by task group for readability
criterion_order = [
    'entity_form', 'type_accuracy', 'hallucination',   # NER
    'relevance', 'style', 'language', 'coverage',      # tags (coverage shared with summary)
    'factual_accuracy', 'specificity',                  # summary
]
criterion_order = [c for c in criterion_order if c in pivot.columns]
pivot = pivot[criterion_order]

# Order models alphabetically
model_order = sorted(pivot.index.tolist())
pivot = pivot.loc[model_order]

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=1, zmax=5,
    text=pivot.round(1).values,
    texttemplate='%{text}',
    hovertemplate='model: %{y}<br>criterion: %{x}<br>avg score: %{z:.2f}<extra></extra>',
))
fig.update_layout(
    title='Average score per model × criterion (across both docs)',
    xaxis_title='Criterion',
    yaxis_title='Model',
    height=420,
    margin=dict(l=140),
)
fig.show()

In [3]:
# Grouped bar chart: mean score per model, faceted by task
task_mean = (
    df.groupby(['model', 'task'])['score']
    .mean()
    .reset_index()
    .rename(columns={'score': 'mean_score'})
)

# Order models alphabetically
model_order = sorted(df['model'].unique().tolist())

fig = px.bar(
    task_mean,
    x='model',
    y='mean_score',
    facet_col='task',
    category_orders={'model': model_order, 'task': ['ner', 'tags', 'summary']},
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    labels={'mean_score': 'Mean score', 'model': 'Model'},
    title='Mean score per task and model (averaged across both docs and task criteria)',
    height=400,
)
fig.update_layout(xaxis_tickangle=-40, coloraxis_showscale=False)
fig.update_yaxes(range=[0, 5.2])
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1].upper()))
fig.show()

In [4]:
# Scatter: mean elapsed_s vs mean quality score (speed / quality tradeoff)
agg = (
    df.groupby('model')
    .agg(mean_score=('score', 'mean'), mean_elapsed=('elapsed_s', 'mean'))
    .reset_index()
)

fig = px.scatter(
    agg,
    x='mean_elapsed',
    y='mean_score',
    text='model',
    labels={'mean_elapsed': 'Mean elapsed (s)', 'mean_score': 'Mean score'},
    title='Speed vs quality tradeoff per model',
    color='mean_score',
    color_continuous_scale='RdYlGn',
    range_color=[1, 5],
    height=450,
)
fig.update_traces(textposition='top center', marker_size=10)
fig.update_layout(yaxis_range=[1, 5.2], coloraxis_showscale=False)
fig.show()

## Conclusions - Grader: claude-haiku-4-5

**Best model: llama3.1:8b and granite3.3:8b (tied, 4.32/5)**
**2nd best: gemma2:2b (4.23/5)**

### Task means
| Task | Mean |
|------|------|
| Tags | 4.39 |
| Summary | 4.38 |
| NER | 3.08 |

### Key findings

**NER is the universal weak point.** Every model struggles: `entity_form` (2.55) is the weakest criterion across the entire eval; all models retain honorifics (`Dr.`, `Professor`) and context phrases (`Danish wind energy company Ørsted`). `type_accuracy` (3.20) is the next gap; PROD is systematically overused for organisations and policy instruments.

**Two models fail NER entirely.** `deepseek-r1:8b` and `qwen3:8b` score 1.00 on NER due to JSON parse failures on both documents. Both perform adequately on tags and summary.

**Tags and summary are consistently strong.** Relevance near-universal (4.70). The only tags gap is `style` (3.95); models produce compound descriptors instead of atomic terms. Summary `specificity` (3.85) is the only moderate shortfall; key figures are usually present but secondary storylines drop out.

**Haiku grades `factual_accuracy` too leniently.** Every one of the 60 records received 5.00; the criterion offers no differentiation. Cross-reference with sonnet and opus graders for a more discriminating view.

### NER ranking (functional models only)
qwen2.5:14b (4.33) > llama3.1:8b (4.17) > mistral:7b (4.00) > gemma2:2b (3.83)